# Grounded Prompting
- RAG 답변의 근거 범위를 명확히 제한하는 방법

In [5]:
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = 'gpt-4.1-mini'

## 실습용 문서

In [1]:
from langchain_core.documents import Document

DOCS = [
    Document(
        page_content='RAG는 사용자의 질문과 관련 있는 외부 문서를 검색한 뒤, 검색된 문서를 LLM의 context로 넣어 답변을 생성하는 방식이다.',
        metadata={'doc_id': 'G01', 'title': 'RAG 기본 개념'}
    ),
    Document(
        page_content='RAG의 검색 단계에서는 사용자의 질문과 관련 있는 문서를 벡터 DB나 검색 시스템에서 찾는다.',
        metadata={'doc_id': 'G02', 'title': 'RAG 검색 단계'}
    ),
    Document(
        page_content='RAG의 생성 단계에서는 검색된 문서를 prompt의 context로 제공하고, LLM이 그 문서를 근거로 답변을 생성한다.',
        metadata={'doc_id': 'G03', 'title': 'RAG 생성 단계'}
    ),
]

retrieved_docs = DOCS
retrieved_docs

[Document(metadata={'doc_id': 'G01', 'title': 'RAG 기본 개념'}, page_content='RAG는 사용자의 질문과 관련 있는 외부 문서를 검색한 뒤, 검색된 문서를 LLM의 context로 넣어 답변을 생성하는 방식이다.'),
 Document(metadata={'doc_id': 'G02', 'title': 'RAG 검색 단계'}, page_content='RAG의 검색 단계에서는 사용자의 질문과 관련 있는 문서를 벡터 DB나 검색 시스템에서 찾는다.'),
 Document(metadata={'doc_id': 'G03', 'title': 'RAG 생성 단계'}, page_content='RAG의 생성 단계에서는 검색된 문서를 prompt의 context로 제공하고, LLM이 그 문서를 근거로 답변을 생성한다.')]

## Context 문자열 만들기
- Document -> LLM prompt에 넣을 때 가공하는 함수

In [3]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

context = format_docs(retrieved_docs)
print(context)

RAG는 사용자의 질문과 관련 있는 외부 문서를 검색한 뒤, 검색된 문서를 LLM의 context로 넣어 답변을 생성하는 방식이다.

RAG의 검색 단계에서는 사용자의 질문과 관련 있는 문서를 벡터 DB나 검색 시스템에서 찾는다.

RAG의 생성 단계에서는 검색된 문서를 prompt의 context로 제공하고, LLM이 그 문서를 근거로 답변을 생성한다.


## 최종 결과를 만들 LLM

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=LLM_MODEL,temperature=0)
output_parser = StrOutputParser()

## 나쁜 예

In [7]:
loose_prompt = PromptTemplate.from_template("""
다음 문서를 참고하여 질문에 자연스럽게 답변하세요.
문서에 없는 내용이더라도 알고 있는 일반 지식을 함께 사용할 수 있습니다.
                                            
[Context]
{context}
                                            
[Question]
{question}
                                            
[Answer]
""")

loose_chain = loose_prompt | llm | output_parser

loose_chain.invoke({
    'context':context,
    'question':'RAG를 처음 제안한 논문의 저자는 누구인가요?'
})

'RAG(Retrieval-Augmented Generation) 방식을 처음 제안한 논문은 페이스북 AI 연구팀(Facebook AI Research, FAIR)에서 발표한 것으로, 주요 저자 중 한 명은 Patrick Lewis입니다. 이 논문에서는 외부 문서를 검색해 LLM의 컨텍스트로 활용하는 방법을 소개하며, 검색과 생성 단계를 결합한 새로운 접근법을 제안했습니다.'

## Grounded Prompt로 개선하기
- 답변 규칙을 명확하게 작성한다.
- RAG에서 일반적으로 포함 되면 좋을 규칙
    - [Context]에 있는 내용만 근거로 사용하세요.
    - 질문에 직접 답하는 내용만 작성하세요.
    - 문서에 없는 내용은 추측하지 마세요.
    - 질문에 답할 근거가 부족하면 "제공된 문서만으로는 답변할 수 없습니다."라고 답하세요.
    - 답변은 3문장 이내로 작성하세요.

In [11]:
grounded_prompt = PromptTemplate.from_template("""
당신은 검색된 문서를 바탕으로 답변하는 RAG assistant입니다.

규칙:
1. [Context]에 있는 내용만 근거로 사용하세요.
2. 질문에 직접 답하는 내용만 작성하세요.
3. 문서에 없는 내용은 추측하지 마세요.
4. 질문에 답할 근거가 부족하면 "제공된 문서만으로는 답변할 수 없습니다."라고 답하세요.
5. 답변은 3문장 이내로 작성하세요.
                                            
[Context]
{context}
                                            
[Question]
{question}
                                            
[Answer]
""")

grounded_chain = grounded_prompt | llm | output_parser

grounded_chain.invoke({
    'context':context,
    'question':'RAG를 처음 제안한 논문의 저자는 누구인가요?'
})

'제공된 문서만으로는 답변할 수 없습니다.'

## 여러 질문을 통해 태스트

In [12]:
questions = [
    'RAG의 검색 단계에서는 무엇을 하나요?',
    'RAG의 생성 단계에서는 무엇을 하나요?',
    'RAG를 처음 제안한 논문의 제목은 무엇인가요?',
    'RAG와 Fine-tuning의 차이는 무엇인가요?',
]

for question in questions:
    answer = grounded_chain.invoke({
        'context':context,
        'question':question
    })
    print(f'질문:{question}')
    print(f'답변:{answer}')
    print('='*100)

질문:RAG의 검색 단계에서는 무엇을 하나요?
답변:RAG의 검색 단계에서는 사용자의 질문과 관련 있는 문서를 벡터 DB나 검색 시스템에서 찾습니다.
질문:RAG의 생성 단계에서는 무엇을 하나요?
답변:RAG의 생성 단계에서는 검색된 문서를 prompt의 context로 제공하고, LLM이 그 문서를 근거로 답변을 생성합니다.
질문:RAG를 처음 제안한 논문의 제목은 무엇인가요?
답변:제공된 문서만으로는 답변할 수 없습니다.
질문:RAG와 Fine-tuning의 차이는 무엇인가요?
답변:RAG는 외부 문서를 검색해 그 문서를 기반으로 답변을 생성하는 방식이고, Fine-tuning은 LLM 자체를 특정 데이터로 추가 학습시키는 방법입니다. RAG는 검색된 문서를 context로 사용하지만, Fine-tuning은 모델의 내부 파라미터를 조정합니다. 제공된 문서에는 Fine-tuning에 대한 구체적 설명은 없으나, RAG는 검색과 생성의 두 단계를 포함합니다.
